# Designing Graph Datasets with TopologicPy  
## From Spatial Models to Trainable Graph Data

This notebook is a **single end-to-end tutorial** built around one small spatial example and one target task.

## Learning task
Predict whether a room is a **circulation space** or not.

## Dataset design choices
- **One graph = one floor**
- **Nodes = rooms**
- **Edges = direct adjacency**
- **Target = `is_circulation_space`**
- **Feature families = geometry + topology + semantics**

The notebook uses **TopologicPy** to create a simple architectural model, then prepares an ML-ready graph representation and exports it to PyTorch Geometric ready CSV files.


In [1]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [2]:
import math
import numpy as np
import pandas as pd
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [3]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.18) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 4. Define the task

We begin with the task, not the file format.

### Task
**Node classification**: predict whether a room is a circulation space.

This determines:
- what counts as a node
- what counts as an edge
- what features are useful
- what the target should be


## 5. Create a small synthetic building in TopologicPy

We construct six rooms:
- one entrance
- one corridor
- two offices
- one meeting room
- one WC

To keep the tutorial robust and transparent, we store both:
1. A **TopologicPy cell** for each room
2. A parallel **metadata record** used for feature engineering


In [5]:
def make_room(x, y, width, length, height, name, room_type):
    origin = Vertex.ByCoordinates(x, y, 0)
    face = Face.Rectangle(origin=origin, width=width, length=length, placement="lowerleft")
    cell = Cell.ByThickenedFace(face, thickness=height)

    d = Dictionary.ByKeysValues(
        ["name", "room_type", "x", "y", "width", "length", "height"],
        [name, room_type, x, y, width, length, height]
    )
    Topology.SetDictionary(cell, d)

    record = {
        "name": name,
        "room_type": room_type,
        "x": x,
        "y": y,
        "width": width,
        "length": length,
        "height": height,
        "cell": cell
    }
    return record

rooms = [
    make_room(0, 3, 1, 1, 3, "entrance", "circulation"),
    make_room(1, 3, 8, 1, 3, "corridor", "circulation"),
    make_room(1, 4, 4, 3, 3, "office_1", "office"),
    make_room(5, 4, 4, 3, 3, "office_2", "office"),
    make_room(1, 0, 6, 3, 3, "meeting", "meeting"),
    make_room(7, 0, 2, 3, 3, "wc", "wc"),
]

rooms_df = pd.DataFrame([{k: v for k, v in r.items() if k != "cell"} for r in rooms])
rooms_df


,name,room_type,x,y,width,length,height
0,entrance,circulation,0,3,1,1,3
1,corridor,circulation,1,3,8,1,3
2,office_1,office,1,4,4,3,3
3,office_2,office,5,4,4,3,3
4,meeting,meeting,1,0,6,3,3
5,wc,wc,7,0,2,3,3


## 6. Build the CellComplex of the Building to Derive the Adjacency Graph

In [6]:
cells = [d['cell'] for d in rooms]
building = CellComplex.ByCells(cells, transferDictionaries=True, silent=True)
print(building)

## 7. Show the geometry

In [7]:
# For proper visualisation we will place a vertex at the centroid to show the room name

centroids = []
for cell in cells:
    centroid = Topology.Centroid(cell)
    d = Topology.Dictionary(cell)
    d = Dictionary.SetValueAtKey(d, "size", 0.00001) # very small vertex size so it disappears
    centroid = Topology.SetDictionary(centroid, d)
    centroids.append(centroid)

Topology.Show(building, centroids,
              vertexSizeKey="size",
              vertexColorKey="color",
              showVertexLabel = True,
              vertexLabelKey="name",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceColor=[210,210,250],
              faceOpacity=0.07,
              width=800,
              height=600,
              backgroundColor="white",
              renderer= renderer)

## 8. Check the dictionaries of the cells

In [8]:
cc_cells = Topology.Cells(building)
for c in cc_cells:
    d = Topology.Dictionary(c)
    print(Dictionary.Keys(d), Dictionary.Values(d))


['aabb', 'height', 'length', 'name', 'room_type', 'width', 'x', 'y'] [[0.0, 3.0, -1.5, 1.0, 4.0, 1.5], 3, 1, 'entrance', 'circulation', 1, 0, 3]
['aabb', 'height', 'length', 'name', 'room_type', 'width', 'x', 'y'] [[1.0, 3.0, -1.5, 9.0, 4.0, 1.5], 3, 1, 'corridor', 'circulation', 8, 1, 3]
['aabb', 'height', 'length', 'name', 'room_type', 'width', 'x', 'y'] [[1.0, 0.0, -1.5, 7.0, 3.0, 1.5], 3, 3, 'meeting', 'meeting', 6, 1, 0]
['aabb', 'height', 'length', 'name', 'room_type', 'width', 'x', 'y'] [[7.0, 0.0, -1.5, 9.0, 3.0, 1.5], 3, 3, 'wc', 'wc', 2, 7, 0]
['aabb', 'height', 'length', 'name', 'room_type', 'width', 'x', 'y'] [[5.0, 4.0, -1.5, 9.0, 7.0, 1.5], 3, 3, 'office_2', 'office', 4, 5, 4]
['aabb', 'height', 'length', 'name', 'room_type', 'width', 'x', 'y'] [[1.0, 4.0, -1.5, 5.0, 7.0, 1.5], 3, 3, 'office_1', 'office', 4, 1, 4]


## 9. Derive the Adjacency Graph

In [9]:
g = Graph.ByTopology(building, storeBREP=True)
g_verts = Graph.Vertices(g)
g_edges = Graph.Edges(g)

## 10. Assign Visual Attributes for Graph vertices and edges

In [10]:
for v in g_verts:
    d = Topology.Dictionary(v)
    d = Dictionary.SetValuesAtKeys(d, ["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
for e in g_edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
    e = Topology.SetDictionary(e, d)

## 10. Show the building and the graph

We now visualise:
- room outlines
- room labels
- adjacency connections between room centroids

This is useful both pedagogically and for dataset debugging.


In [11]:
Topology.Show(building, g,
              vertexSizeKey="size",
              vertexColorKey="color",
              showVertexLabel = True,
              vertexLabelKey="name",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceColor=[210,210,250],
              faceOpacity=0.07,
              width=800,
              height=600,
              backgroundColor="white",
              renderer= renderer)


## 11. Define the target/label

The target is binary:

- `1` if the room is a circulation space
- `0` otherwise

In a real dataset, this target would come from a well-defined annotation protocol. Here we derive it directly from the synthetic room type.


In [12]:
def get_target(d):
    return 1 if Dictionary.ValueAtKey(d,"room_type") == "circulation" else 0


targets = []
for v in g_verts:
    d = Topology.Dictionary(v)
    target = get_target(d)
    d = Dictionary.SetValueAtKey(d, "label", target)
    v = Topology.SetDictionary(v, d)
    targets.append(target)
y = np.array(targets, dtype=int)
y


array([1, 0, 1, 0, 0, 0])

## 12. Engineer geometric features

We compute a minimal geometric feature set:
- area
- perimeter
- aspect ratio
- compactness

This is deliberately small. The goal is to start with a clear, defensible schema.


In [13]:
def geometric_features(d):
    py_d = Dictionary.PythonDictionary(d)
    area = py_d["width"] * py_d["length"]
    perimeter = 2 * (py_d["width"] + py_d["length"])
    aspect_ratio = max(py_d["width"], py_d["length"]) / min(py_d["width"], py_d["length"])
    compactness = area / (perimeter ** 2) if perimeter > 0 else 0.0
    return {
        "area": area,
        "perimeter": perimeter,
        "aspect_ratio": aspect_ratio,
        "compactness": compactness
    }

g_features = []
for v in g_verts:
    d = Topology.Dictionary(v)
    geo_feat = geometric_features(d)
    for key in geo_feat:
        d = Dictionary.SetValueAtKey(d, key, geo_feat[key])
    v = Topology.SetDictionary(v, d)
    g_features.append(geo_feat)
geom_df = pd.DataFrame(g_features)
geom_df.index = [room["name"] for room in rooms]
geom_df


,area,perimeter,aspect_ratio,compactness
entrance,8,18,8.000000,0.024691
corridor,18,18,2.000000,0.055556
office_1,1,4,1.000000,0.062500
office_2,12,14,1.333333,0.061224
meeting,12,14,1.333333,0.061224
wc,6,10,1.500000,0.060000


## 13. Engineer topological features

We compute graph-based features from the adjacency structure:
- degree
- topological distance to entrance (number of hops/edges)
- is directly connected to entrance


In [14]:
entrance = None
for v in g_verts:
    d = Topology.Dictionary(v)
    if Dictionary.ValueAtKey(d, "name") == "entrance":
        entrance = v
    degree = Graph.VertexDegree(g, v)
    d = Dictionary.SetValuesAtKeys(d, ["degree", "is_connected_to_entrance"], [degree, 0])
    v = Topology.SetDictionary(v, d)

for v in g_verts:
    d = Topology.Dictionary(v)
    path = Graph.ShortestPath(g, v, entrance)
    if Topology.IsInstance(path, "wire"):
        dist = len(Topology.Edges(path)) # Topological Distance
    elif Topology.IsInstance(path, "edge"):
        dist = 1
    else:
        dist = 0
    d = Dictionary.SetValueAtKey(d, "distance_to_entrance", dist)
    v = Topology.SetDictionary(v, d)

adj_verts = Graph.AdjacentVertices(g, entrance)

for adj_vert in adj_verts:
    d = Topology.Dictionary(adj_vert)
    d = Dictionary.SetValueAtKey(d, "is_connected_to_entrance", 1)
    adj_verts = Topology.SetDictionary(adj_vert, d)

def topological_features(d):
    return {
        "degree": Dictionary.ValueAtKey(d, "degree"),
        "distance_to_entrance": Dictionary.ValueAtKey(d, "distance_to_entrance"),
        "is_connected_to_entrance": Dictionary.ValueAtKey(d, "is_connected_to_entrance")
    }

topo_df = pd.DataFrame([topological_features(Topology.Dictionary(v)) for v in g_verts])
topo_df.index = [Dictionary.ValueAtKey(Topology.Dictionary(v), "name") for v in g_verts]
topo_df


,degree,distance_to_entrance,is_connected_to_entrance
corridor,5,1,1
meeting,2,2,0
entrance,1,0,0
office_2,2,2,0
office_1,2,2,0
wc,2,2,0


## 14. Engineer semantic features

We use one nominal semantic feature:
- `room_type`

This must **not** be encoded as arbitrary integers because the categories have no intrinsic order.


In [15]:
room_types = sorted({room["room_type"] for room in rooms})
room_types


['circulation', 'meeting', 'office', 'wc']

In [16]:
def one_hot_encode(value, categories):
    return [1 if value == cat else 0 for cat in categories]

# This must **not** be encoded as arbitrary integers because the categories have no intrinsic order.
room_types = sorted({room["room_type"] for room in rooms})
keys = ["room_type_"+s for s in room_types]

for v in g_verts:
    d = Topology.Dictionary(v)
    room_type = Dictionary.ValueAtKey(d, "room_type")
    ohc = one_hot_encode(room_type, room_types)
    d = Dictionary.SetValuesAtKeys(d, keys, ohc)
    v = Topology.SetDictionary(v, d)


def semantic_features(d):
    return_dict = {
    "room_type": Dictionary.ValueAtKey(d, "room_type")
    }
    for key in keys:
        return_dict[key] = Dictionary.ValueAtKey(d, key)
   
    return return_dict

semantic_df = pd.DataFrame([semantic_features(Topology.Dictionary(v)) for v in g_verts])
semantic_df.index = [Dictionary.ValueAtKey(Topology.Dictionary(v), "name") for v in g_verts]
semantic_df


,room_type,room_type_circulation,room_type_meeting,room_type_office,room_type_wc
corridor,circulation,1,0,0,0
meeting,meeting,0,1,0,0
entrance,circulation,1,0,0,0
office_2,office,0,0,1,0
office_1,office,0,0,1,0
wc,wc,0,0,0,1


## 15. Assemble the feature matrix

We now combine:
- geometric features
- topological features
- one-hot semantic features

We also create a feature schema table so the dataset is explicit and reproducible.


In [17]:
feature_df = pd.concat(
    [
        geom_df.reset_index(drop=True),
        topo_df.reset_index(drop=True),
        semantic_df.drop(columns=["room_type"]).reset_index(drop=True)
    ],
    axis=1
)
feature_df.index = [Dictionary.ValueAtKey(Topology.Dictionary(v), "name") for v in g_verts]
feature_df


,area,perimeter,aspect_ratio,compactness,degree,distance_to_entrance,is_connected_to_entrance,room_type_circulation,room_type_meeting,room_type_office,room_type_wc
corridor,8,18,8.000000,0.024691,5,1,1,1,0,0,0
meeting,18,18,2.000000,0.055556,2,2,0,0,1,0,0
entrance,1,4,1.000000,0.062500,1,0,0,1,0,0,0
office_2,12,14,1.333333,0.061224,2,2,0,0,0,1,0
office_1,12,14,1.333333,0.061224,2,2,0,0,0,1,0
wc,6,10,1.500000,0.060000,2,2,0,0,0,0,1


In [18]:
schema_rows = [
    ("area", "node", "continuous", "m2", "raw", "room width × room length", True),
    ("perimeter", "node", "continuous", "m", "raw", "2 × (width + length)", True),
    ("aspect_ratio", "node", "continuous", "-", "raw", "max(width, length) / min(width, length)", True),
    ("compactness", "node", "continuous", "-", "raw", "area / perimeter^2", True),
    ("degree", "node", "continuous", "count", "raw", "adjacency degree", True),
    ("distance_to_entrance", "node", "continuous", "steps", "raw", "Shortest Path to Entrance", True),
    ("is_connected_to_entrance", "node", "binary", "0/1", "raw", "adjacent to corridor", True),
]
for cat in room_types:
    schema_rows.append((
        f"room_type__{cat}",
        "node",
        "categorical",
        "0/1",
        "one-hot",
        "one-hot encoding of room_type",
        True
    ))

schema_df = pd.DataFrame(
    schema_rows,
    columns=[
        "feature_name",
        "entity_type",
        "data_type",
        "units",
        "encoding_method",
        "source_method",
        "available_at_inference"
    ]
)
schema_df


,feature_name,entity_type,data_type,units,encoding_method,source_method,available_at_inference
0,area,node,continuous,m2,raw,room width × room length,True
1,perimeter,node,continuous,m,raw,2 × (width + length),True
2,aspect_ratio,node,continuous,-,raw,"max(width, length) / min(width, length)",True
3,compactness,node,continuous,-,raw,area / perimeter^2,True
4,degree,node,continuous,count,raw,adjacency degree,True
5,distance_to_entrance,node,continuous,steps,raw,Shortest Path to Entrance,True
6,is_connected_to_entrance,node,binary,0/1,raw,adjacent to corridor,True
7,room_type__circulation,node,categorical,0/1,one-hot,one-hot encoding of room_type,True
8,room_type__meeting,node,categorical,0/1,one-hot,one-hot encoding of room_type,True
9,room_type__office,node,categorical,0/1,one-hot,one-hot encoding of room_type,True


In [19]:
node_features = feature_df.columns.tolist()
# Remove leaky columns
node_features = [c for c in node_features if "room_type_" not in c]
print(node_features)

['area', 'perimeter', 'aspect_ratio', 'compactness', 'degree', 'distance_to_entrance', 'is_connected_to_entrance']


## 16. Export Graph to PyTorch Geometric compatible CSV files


In [20]:
status = Graph.ExportToCSV(g,
                           path = r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset",
                           nodeLabelKey="label",
                           nodeFeaturesKeys = node_features,
                           overwrite=True)
print(status)

True


## 18. Final checklist

Before training a model, ask:

- Is the graph definition aligned with the task?
- Are the nodes comparable across the dataset?
- Are the edge semantics consistent?
- Are the features available at inference time?
- Have nominal categories been encoded correctly?
- Is there leakage in preprocessing, splitting, or features?
- Are train and test samples genuinely distinct?
- Is the feature schema documented?

## Conclusion

This notebook showed an end-to-end path from:
**spatial model → graph definition → feature schema → encoded dataset → ML-ready CSV file**

The key point is that good graph learning begins with disciplined dataset design.
